In [1]:
import argparse
import json
import math
import sys
from collections import defaultdict
from typing import Dict, List, Sequence, Tuple

import numpy as np
import torch
from sae_lens import SAE, HookedSAETransformer
from scipy.stats import mannwhitneyu  # pip install scipy

sys.path.append("../")
from plan_trace.utils import load_model, load_pretrained_saes, cleanup_cuda
from plan_trace.hooks import run_with_saes, register_sae_hooks

In [2]:
# -----------------------------
# Helpers
# -----------------------------


def tokens_to_text(tokens: Sequence[str]) -> str:
    """Reconstruct text from Neuronpedia-style token strings."""
    return "".join(
        t.replace("▁", " ").replace("<0x0A>", "\n") for t in tokens
    )


def decode_token_id(model: HookedSAETransformer, token_id: int) -> str:
    """Decode a single token id to a readable string."""
    try:
        return model.tokenizer.decode([token_id])
    except Exception:
        return f"<id:{token_id}>"


def compute_top_q_summary(
    latent_acts: torch.Tensor,
    top_q: float,
) -> torch.Tensor:
    """
    latent_acts: [B, L] tensor of latent activations per position.
    top_q: fraction (0 < top_q <= 1). E.g. 0.01 for top 1%.

    Returns: [B] sequence-level summaries (mean of top-q activations).
    """
    B, L = latent_acts.shape
    k = max(1, int(math.ceil(L * top_q)))
    top_vals, _ = torch.topk(latent_acts, k=k, dim=1)
    return top_vals.mean(dim=1)


def build_token_to_seq_index(
    all_token_ids: List[List[int]],
    special_ids: Sequence[int],
) -> Dict[int, List[int]]:
    """
    Build inverted index: token_id -> list of sequence indices where it appears.
    Presence is sequence-level (once per sequence).
    """
    special_set = set(special_ids)
    token_to_seqs: Dict[int, List[int]] = defaultdict(list)

    for seq_idx, ids in enumerate(all_token_ids):
        seen = set()
        for tid in ids:
            if tid in special_set:
                continue
            if tid in seen:
                continue
            seen.add(tid)
            token_to_seqs[tid].append(seq_idx)

    return token_to_seqs

In [3]:
# setup stuff
prompts_path = "../data/prompts.json"
num_seqs = 10000
model_name = "gemma-2-2b"
device = "cuda"
sae_release = "gemma-scope-2b-pt-mlp"
latent_layer = 7
latent_index = 7643

In [5]:
with open(prompts_path, "r") as f:
    prompts = json.load(f)

if num_seqs > len(prompts):
    print(f"[WARN] Requested num_seqs={num_seqs} > available={len(prompts)}; using all.")
    num_seqs = len(prompts)

prompts = prompts[:num_seqs]
print(f"[INFO] Loaded {len(prompts)} prompts from {prompts_path}")

# Convert token lists -> raw text strings
texts: List[str] = [tokens_to_text(p) for p in prompts]

# 2. Load model (Gemma-2-2B)
print(f"[INFO] Loading model: {model_name}")
model: HookedSAETransformer = load_model(
    model_name,
    device=device,
    use_custom_cache=True,
    dtype=torch.bfloat16,
)

# 3. Load SAE for the desired layer
print(f"[INFO] Loading SAE: release={sae_release}")
# sae, cfg_dict, sparsity = SAE.from_pretrained(
#     release=sae_release,
#     sae_id=sae_id,
#     device=device,
# )
from sae_lens import SAE
sae, cfg_dict, sparsity = SAE.from_pretrained(release="gemma-scope-2b-pt-mlp-canonical", sae_id="layer_7/width_16k/canonical", device=device)

[INFO] Loaded 10000 prompts from ../data/prompts.json


[INFO] Loading model: gemma-2-2b


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model gemma-2-2b into HookedTransformer
[INFO] Loading SAE: release=gemma-scope-2b-pt-mlp


In [6]:
sae.cfg

SAEConfig(architecture='jumprelu', d_in=2304, d_sae=16384, activation_fn_str='relu', apply_b_dec_to_input=False, finetuning_scaling_factor=False, context_size=1024, model_name='gemma-2-2b', hook_name='blocks.7.hook_mlp_out', hook_layer=7, hook_head_index=None, prepend_bos=True, dataset_path='monology/pile-uncopyrighted', dataset_trust_remote_code=True, normalize_activations=None, dtype='float32', device='cuda', sae_lens_training_version=None, activation_fn_kwargs={}, neuronpedia_id='gemma-2-2b/7-gemmascope-mlp-16k', model_from_pretrained_kwargs={}, seqpos_slice=(None,))

In [7]:
sae.W_dec.shape

torch.Size([16384, 2304])

In [11]:
top_q = 0.01  # top 1%
batch_size = 32
min_freq = 20
max_freq_frac = 0.9

In [9]:
text = """="searchByAge"/>
▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁<br▁/><br▁/>
▁▁▁▁▁▁▁▁▁▁▁▁</div>
▁▁▁▁▁▁▁▁▁▁▁▁</form>

And▁want▁to▁show▁the▁result▁in▁console.log:
$('.searchByAge').on('click',▁function(e){
e.preventDefault();

var▁range▁=▁$('.form-control').val();
var▁min▁=▁range.split(',')[0];
var▁max▁=▁range.split(',')[1];

//alert(min+'▁'+max);

$.ajax({
▁▁▁▁type:▁'POST',
▁▁▁▁url:▁'/age/'+min+'/'+max,
▁▁▁▁data:▁$('.searchForm').serialize"""
tokens = model.to_tokens(text).to(device)
str_tokens = model.to_str_tokens(tokens)
with torch.no_grad():
    model.reset_hooks(including_permanent=True)
    _ = run_with_saes(model, saes=[sae], tokens=tokens, cache_sae_activations=True)
    feature_acts = sae.feature_acts
    latent_acts = feature_acts[..., latent_index] 


In [10]:
for i in range(len(str_tokens)):
    print(f"{i:4d}: {str_tokens[i]:15s} | {latent_acts[0, i].item():.4f}")

   0: <bos>           | 0.0000
   1: ="              | 0.0000
   2: search          | 0.0000
   3: By              | 0.0000
   4: Age             | 0.0000
   5: "/>             | 0.0000
   6: 
               | 0.0000
   7:                  | 0.0000
   8: <               | 0.0000
   9: br              | 0.0000
  10:  /><            | 0.0000
  11: br              | 0.0000
  12:  />             | 0.0000
  13: 
               | 0.0000
  14:                 | 0.0000
  15: </              | 0.0000
  16: div             | 0.0000
  17: >               | 0.0000
  18: 
               | 0.0000
  19:                 | 0.0000
  20: </              | 0.0000
  21: form            | 0.0000
  22: >               | 0.0000
  23: 

              | 0.0000
  24: And             | 0.0000
  25:  want           | 0.0000
  26:  to             | 0.0000
  27:  show           | 0.0000
  28:  the            | 0.0000
  29:  result         | 0.0000
  30:  in             | 0.0000
  31:  console        | 0.0000
  32: .

In [12]:

# 4. Tokenize prompts + collect token ids + latent summaries
from tqdm import tqdm


print("[INFO] Tokenizing prompts and computing latent activations...")
model.eval()
all_seq_token_ids: List[List[int]] = []
all_seq_summaries: List[torch.Tensor] = []

# Try to get special token ids to ignore
special_ids = []
for attr in ("bos_id", "eos_id", "pad_id"):
    if hasattr(model.tokenizer, attr):
        v = getattr(model.tokenizer, attr)
        if v is not None:
            special_ids.append(int(v))

with torch.no_grad():
    for start in tqdm(range(0, len(texts), batch_size)):
        end = min(start + batch_size, len(texts))
        batch_texts = texts[start:end]

        # [B, L]
        tokens = model.to_tokens(batch_texts).to(device)
        B, L = tokens.shape

        # Run model with SAE, cache feature activations
        model.reset_hooks(including_permanent=True)
        _ = run_with_saes(model, saes=[sae], tokens=tokens, cache_sae_activations=True)

        # sae.feature_acts: [B, L, D_sae]
        feature_acts = sae.feature_acts  # type: ignore
        if feature_acts is None:
            raise RuntimeError("SAE feature_acts not populated; check run_with_saes call.")

        latent_acts = feature_acts[..., latent_index]  # [B, L]
        seq_summary_batch = compute_top_q_summary(latent_acts, top_q=top_q)  # [B]

        # Save
        all_seq_summaries.append(seq_summary_batch.cpu())
        all_seq_token_ids.extend(tokens.cpu().tolist())

seq_summaries = torch.cat(all_seq_summaries, dim=0).numpy()  # [N]
N = len(seq_summaries)
assert N == len(all_seq_token_ids)
print(f"[INFO] Got sequence summaries for N={N} sequences")

# 5. Build token -> sequence index mapping, compute frequencies
print("[INFO] Building token->sequence index mapping...")
token_to_seqs = build_token_to_seq_index(all_seq_token_ids, special_ids)
print(f"[INFO] Found {len(token_to_seqs)} distinct non-special tokens")

# 6. Frequency-based token filtering
print(
    f"[INFO] Applying frequency filters: min_freq={min_freq}, "
    f"max_freq_frac={max_freq_frac}"
)
candidates = []
for tid, seq_indices in token_to_seqs.items():
    f = len(seq_indices)
    if f < min_freq:
        continue
    if f > max_freq_frac * N:
        continue
    candidates.append((tid, seq_indices))

print(f"[INFO] {len(candidates)} tokens remain after frequency filtering")


[INFO] Tokenizing prompts and computing latent activations...


  0%|          | 0/313 [00:00<?, ?it/s]

100%|██████████| 313/313 [17:29<00:00,  3.35s/it]


[INFO] Got sequence summaries for N=10000 sequences
[INFO] Building token->sequence index mapping...
[INFO] Found 57702 distinct non-special tokens
[INFO] Applying frequency filters: min_freq=20, max_freq_frac=0.9
[INFO] 4845 tokens remain after frequency filtering


In [13]:
alpha = 0.05
max_print = 50

In [14]:

# 7. MWU + AUC per token
all_results: List[Tuple[int, int, float, float]] = []
all_indices = np.arange(N)

print("[INFO] Computing MWU & AUC for each candidate token...")
for tid, pos_seqs in candidates:
    pos_idx = np.array(pos_seqs, dtype=int)
    neg_mask = np.ones(N, dtype=bool)
    neg_mask[pos_idx] = False
    neg_idx = all_indices[neg_mask]

    pos = seq_summaries[pos_idx]
    neg = seq_summaries[neg_idx]

    if len(pos) == 0 or len(neg) == 0:
        continue

    # Mann–Whitney U test
    U, p = mannwhitneyu(pos, neg, alternative="two-sided")

    # AUC: U / (n_pos * n_neg)
    n_pos = len(pos)
    n_neg = len(neg)
    auc = U / (n_pos * n_neg)

    all_results.append((tid, n_pos, auc, p))

if not all_results:
    print("[WARN] No tokens had valid MWU statistics. Try loosening filters.")


[INFO] Computing MWU & AUC for each candidate token...


In [16]:

# 8. Bonferroni correction and ranking
M = len(all_results)
print(f"[INFO] Applying Bonferroni correction over M={M} tests; alpha={alpha}")
corrected_results = []
for tid, freq, auc, p in all_results:
    p_bonf = min(p * M, 1.0)
    corrected_results.append((tid, freq, auc, p, p_bonf))

# Keep only those passing Bonferroni and with non-trivial AUC
filtered = [
    r for r in corrected_results
    if r[4] <= alpha and not math.isnan(r[2])
]
if not filtered:
    print("[WARN] No tokens survived Bonferroni at this alpha. Try relaxing alpha or filters.")
else:
    # Sort by |AUC - 0.5| descending (strongest association first)
    filtered.sort(key=lambda r: abs(r[2] - 0.5), reverse=True)

    # 9. Print results
    print(
        f"\n=== Top tokens for latent (layer={latent_layer}, index={latent_index}) "
        f"===\n"
    )
    print(
        f"{'rank':>4}  {'id':>8}  {'freq':>6}  {'AUC':>8}  {'p_bonf':>10}  token\n"
        + "-" * 80
    )

    for rank, (tid, freq, auc, p_raw, p_bonf) in enumerate(filtered[:200], start=1):
        tok_str = decode_token_id(model, tid)
        tok_str = tok_str.replace("\n", "\\n")
        print(
            f"{rank:4d}  {tid:8d}  {freq:6d}  {auc:8.3f}  {p_bonf:10.2e}  {tok_str}"
        )

print("\n[INFO] Done.")
cleanup_cuda()

[INFO] Applying Bonferroni correction over M=4845 tests; alpha=0.05

=== Top tokens for latent (layer=7, index=7643) ===

rank        id    freq       AUC      p_bonf  token
--------------------------------------------------------------------------------
   1      6593      20     0.711    1.22e-19  vector
   2      4290      31     0.695    1.43e-26  std
   3    255972      21     0.675    1.35e-13  					
   4      2770      22     0.664    2.57e-12   +=
   5      1380      73     0.656    5.29e-41  ];
   6      2968      23     0.637    2.57e-08   max
   7      5329      25     0.622    6.55e-07  mid
   8      3673      22     0.621    1.04e-05  ]);
   9      7114      22     0.620    1.74e-05  toString
  10      1075      67     0.617    2.87e-20  else
  11      6939      26     0.616    2.25e-06  }-
  12      1730      34     0.614    1.51e-08  ';
  13     11298      23     0.614    4.66e-05   parse
  14      1149      28     0.609    7.15e-06  str
  15      2732      40     0.608 

In [44]:
import numpy as np

a = seq_summaries  # [N]
thr = np.quantile(a, 0.96)  # top 10%
y = (a > thr).astype(int)

print("Threshold:", thr)
print("Positive rate:", y.mean())


Threshold: 0.0
Positive rate: 0.0374


In [47]:
print("min, max:", float(a.min()), float(a.max()))
print("percentiles:", np.percentile(a, [50, 75, 90, 95, 99, 99.5, 99.9]))


min, max: 0.0 14.89581298828125
percentiles: [ 0.          0.          0.          0.          1.71489135  2.71954638
 10.820995  ]


In [45]:
K = 200  # you can play with this

# corrected_results: (tid, freq, auc, p_raw, p_bonf)
sorted_pos = sorted(corrected_results, key=lambda r: r[2], reverse=True)
sorted_neg = sorted(corrected_results, key=lambda r: r[2])

primitive_records = sorted_pos[:K] + sorted_neg[:K]
primitive_tids = sorted({r[0] for r in primitive_records})

print("Num primitive tokens:", len(primitive_tids))
decoded_primitives = [decode_token_id(model, tid).replace("\n", "\\n")
                      for tid in primitive_tids]
for tid, tok in zip(primitive_tids, decoded_primitives[:30]):
    print(tid, "->", tok)


Num primitive tokens: 400
475 -> er
494 -> le
536 -> io
549 -> un
573 ->  the
576 ->  of
577 ->  to
578 ->  and
589 ->  =
594 -> );
615 -> end
634 -> //
648 -> if
726 -> /*
801 -> sp
814 -> };
829 ->  */
854 -> red
859 -> });
860 -> */
874 -> row
881 -> set
887 -> able
931 -> so
934 -> val
961 -> form
990 -> ents
1056 -> gen
1059 ->  app
1075 -> else


In [46]:
N = len(seq_summaries)
tid_to_col = {tid: j for j, tid in enumerate(primitive_tids)}
X = np.zeros((N, len(primitive_tids)), dtype=np.int8)

for tid, seq_idxs in token_to_seqs.items():
    if tid not in tid_to_col:
        continue
    j = tid_to_col[tid]
    X[seq_idxs, j] = 1

print("X shape:", X.shape)
print("Mean feature density:", X.mean())


X shape: (10000, 400)
Mean feature density: 0.01453875


In [48]:
K = 500  # you can play with this

order = np.argsort(a)
low_idx = order[:K]
high_idx = order[-K:]

mask = np.zeros_like(a, dtype=bool)
mask[low_idx] = True
mask[high_idx] = True

X_sub = X[mask]
y_sub = np.zeros(2 * K, dtype=int)
y_sub[:K] = 0   # bottom K
y_sub[K:] = 1   # top K

print("X_sub shape:", X_sub.shape)
print("Class balance:", y_sub.mean())


X_sub shape: (1000, 400)
Class balance: 0.5


In [49]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

X_train, X_test, y_train, y_test = train_test_split(
    X_sub, y_sub, test_size=0.3, random_state=0, stratify=y_sub
)

tree = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=20,
    random_state=0,
)
tree.fit(X_train, y_train)

y_pred = tree.predict(X_test)
print("Tree accuracy:", accuracy_score(y_test, y_pred))
print("Tree F1:", f1_score(y_test, y_pred))

feature_names = [decode_token_id(model, tid).replace("\n", "\\n")
                 for tid in primitive_tids]
print(export_text(tree, feature_names=feature_names))


Tree accuracy: 0.55
Tree F1: 0.5846153846153846
|---  and <= 0.50
|   |---  = <= 0.50
|   |   |---  to <= 0.50
|   |   |   |---  of <= 0.50
|   |   |   |   |--- class: 0
|   |   |   |---  of >  0.50
|   |   |   |   |--- class: 1
|   |   |---  to >  0.50
|   |   |   |--- class: 0
|   |---  = >  0.50
|   |   |--- if <= 0.50
|   |   |   |--- ); <= 0.50
|   |   |   |   |--- class: 0
|   |   |   |--- ); >  0.50
|   |   |   |   |--- class: 0
|   |   |--- if >  0.50
|   |   |   |--- class: 0
|---  and >  0.50
|   |--- fig <= 0.50
|   |   |---  = <= 0.50
|   |   |   |---  the <= 0.50
|   |   |   |   |--- class: 0
|   |   |   |---  the >  0.50
|   |   |   |   |--- class: 1
|   |   |---  = >  0.50
|   |   |   |--- class: 0
|   |--- fig >  0.50
|   |   |--- class: 0



In [43]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)

tree = DecisionTreeClassifier(
    max_depth=5,        # keep small so it’s readable
    min_samples_leaf=50,
    random_state=0,
)
tree.fit(X_train, y_train)

print("Tree test accuracy:", tree.score(X_test, y_test))

feature_names = [decode_token_id(model, tid).replace("\n", "\\n")
                 for tid in primitive_tids]

print(export_text(tree, feature_names=feature_names))


Tree test accuracy: 0.97
|--- ]; <= 0.50
|   |--- Fig <= 0.50
|   |   |---  = <= 0.50
|   |   |   |--- $. <= 0.50
|   |   |   |   |--- end <= 0.50
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- end >  0.50
|   |   |   |   |   |--- class: 0
|   |   |   |--- $. >  0.50
|   |   |   |   |--- class: 0
|   |   |---  = >  0.50
|   |   |   |--- [ <= 0.50
|   |   |   |   |--- var <= 0.50
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- var >  0.50
|   |   |   |   |   |--- class: 0
|   |   |   |--- [ >  0.50
|   |   |   |   |--- class: 0
|   |--- Fig >  0.50
|   |   |--- }). <= 0.50
|   |   |   |--- class: 0
|   |   |--- }). >  0.50
|   |   |   |--- class: 0
|--- ]; >  0.50
|   |--- class: 0



In [26]:
len(seq_summaries)

10000